# ⚠️ Pipeline Reset

Notebook này xóa sạch toàn bộ để chạy lại pipeline từ đầu.

**Những gì sẽ bị xóa:**
- Tất cả Iceberg tables (Bronze, Silver, Gold) — catalog metadata
- Tất cả data files & metadata files trên MinIO (`bronze/`, `silver/`, `gold/`)
- Tất cả Spark Streaming checkpoints trên MinIO
- Kafka topic `raw_yelp_users` (xóa + tạo lại)

**Những gì KHÔNG bị xóa:**
- File Yelp gốc tại `/home/jovyan/data/yelp/` — an toàn
- Cấu hình Docker Compose

---
⚠️ **Chỉ chạy khi muốn reset hoàn toàn. Không thể undo.**

> Notebook đã module hóa — toàn bộ logic core nằm trong package `src/`.
> Notebook chỉ setup SparkSession, gọi function, và làm phần interactive (monitor/validate).


In [ ]:
# Xác nhận trước khi reset
confirm = input("Nhập 'RESET' để xác nhận xóa toàn bộ data: ")
if confirm.strip() != "RESET":
    raise SystemExit("❌ Hủy reset — không có gì bị thay đổi")
print("✅ Xác nhận nhận được — bắt đầu reset...")

In [ ]:
import sys
sys.path.append("/home/jovyan")

from src.spark_session import get_spark_session

spark = get_spark_session("Pipeline_Reset")
print("✅ SparkSession ready")

In [ ]:
from src.s3_utils import get_s3_client

s3 = get_s3_client()
print("✅ boto3 S3 client ready")

In [ ]:
# ── BƯỚC 1: Stop tất cả Spark Streaming queries ──────────────────
from src.reset import stop_all_streams

print("BƯỚC 1: Stop Streaming Queries")
print("-" * 40)
stopped = stop_all_streams(spark)
if not stopped:
    print("  Không có stream nào đang chạy")
else:
    for name in stopped:
        print(f"  ⏹  Stopped: {name}")
print("  ✅ Done")

In [ ]:
# ── BƯỚC 2: Drop Iceberg tables khỏi Nessie catalog ──────────────
from src.reset import drop_all_tables

print("BƯỚC 2: Drop Iceberg Tables (Nessie catalog)")
print("-" * 40)
drop_all_tables(spark, include_legacy=True)
print("  ✅ Done")

In [ ]:
# ── BƯỚC 3: Xóa data files & metadata files trên MinIO ───────────
# DROP TABLE chỉ xóa catalog entry, KHÔNG xóa .parquet/.avro/.json trên S3.
from src.reset import clean_minio_data

print("BƯỚC 3: Xóa data/metadata files trên MinIO")
print("-" * 40)
clean_minio_data(s3)
print("  ✅ Done")

In [ ]:
# ── BƯỚC 4: Xóa Spark Streaming Checkpoints trên MinIO ───────────
from src.reset import clean_checkpoints

print("BƯỚC 4: Xóa Checkpoints trên MinIO")
print("-" * 40)
clean_checkpoints(s3, include_legacy=True)
print("  ✅ Done")

In [ ]:
# ── BƯỚC 5: Reset Kafka Topic ─────────────────────────────────────
from src.kafka_utils import reset_topic
from src import config

print("BƯỚC 5: Reset Kafka Topic")
print("-" * 40)
reset_topic(config.KAFKA_TOPIC_YELP_USERS)
print("  ✅ Done")

In [ ]:
# ── BƯỚC 6: Verify toàn bộ đã sạch ──────────────────────────────
from src.reset import verify_clean

print("BƯỚC 6: Verification")
print("-" * 40)
all_ok = verify_clean(spark, s3)

print()
if all_ok:
    print("╔══════════════════════════════════════════╗")
    print("║  ✅ RESET HOÀN TẤT — Sẵn sàng chạy lại  ║")
    print("╚══════════════════════════════════════════╝")
    print()
    print("Thứ tự chạy lại:")
    print("  1. spark_bronze_yelp.ipynb       (chờ log stream đang chạy)")
    print("  2. kafka_producer_yelp.ipynb      (Pass 1 — full load)")
    print("  3. spark_silver_yelp_scd2.ipynb   (start khi Bronze đang consume)")
    print("  4. kafka_producer_yelp.ipynb      (Pass 2 khi Bronze ~2M rows)")
    print("  5. kafka_producer_yelp.ipynb      (Pass 3)")
    print("  6. spark_gold_businesses.ipynb    (batch, sau Silver có expired > 0)")
    print("  7. system_benchmark.ipynb         (chạy sau cùng)")
else:
    print("⚠️  Một số bước chưa sạch — kiểm tra warning ở trên")